<h1><center>NOVA IMS — Information Management School</center></h1>
<h2><center>Text Mining — Spring Semester 2025/2026</center></h2>
<h2><center>Final Pipeline: Market Sentiment Classification of Financial Tweets</center></h2>

<center><b>Group 12</b></center>

| Name | Student № |
|------|-----------|
| João Cardoso | 20240529 |
| Simon Sazonov | 20221689 |
| Artem Polikarpov | 20250443 |


## About this notebook

The single final pipeline: **fine-tuned twitter-RoBERTa**, the best of 29 configurations evaluated in `tm_tests_12_v2.ipynb` (validation macro-F1 0.853; internal-test macro-F1 0.832, scored once). This notebook retrains that model on all 9,543 labelled tweets and produces the submission file `pred_12.csv`. It runs end-to-end without manual steps; a GPU is used automatically when available, and the trained model is cached so re-runs skip training.


## 0. Setup


In [1]:
import os, random
import numpy as np
import pandas as pd

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv
load_dotenv()

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR  = "./data/"
CHECKPOINT = "cardiffnlp/twitter-roberta-base-sentiment-latest"
FINAL_DIR  = "cache/final_model"
LABELS = [0, 1, 2]
LABEL_NAMES = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
print("device:", DEVICE)


device: cuda


## 1. Data Import


In [2]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
assert "id" in test_df.columns, f"expected an 'id' column in test.csv; got {list(test_df.columns)}"

# Row filters before any training (none trigger on this corpus — verified in the experiments notebook)
train_df = (train_df.dropna(subset=["text"])
            .loc[lambda d: d["text"].astype(str).str.strip() != ""]
            .drop_duplicates(subset=["text"]).reset_index(drop=True))
print(f"train: {train_df.shape}  |  test: {test_df.shape}")


train: (9543, 2)  |  test: (2388, 2)


## 2. Final Model — Fine-tuned twitter-RoBERTa

Configuration selected in the experiments notebook: learning rate 2e-5, weight decay 0.01, batch 32, `max_length` 64, and a fixed budget of 4 epochs — the epoch count chosen by per-epoch validation there. Trained here on all 9,543 labelled tweets (no holdout: model selection is already done).


In [3]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=64)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

def train_final_model():
    tok = AutoTokenizer.from_pretrained(CHECKPOINT)
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=3)
    args = TrainingArguments(
        output_dir=FINAL_DIR, num_train_epochs=4, learning_rate=2e-5,
        per_device_train_batch_size=32, weight_decay=0.01,
        save_strategy="no", logging_strategy="epoch", report_to=[],
        seed=RANDOM_STATE, use_cpu=not torch.cuda.is_available())
    trainer = Trainer(model=model, args=args,
                      train_dataset=TweetDataset(train_df["text"], train_df["label"], tok))
    trainer.train()
    model.save_pretrained(FINAL_DIR); tok.save_pretrained(FINAL_DIR)

if os.path.exists(os.path.join(FINAL_DIR, "model.safetensors")):
    print("[cache] final model found — training skipped")
else:
    train_final_model()


[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
299,0.486286
598,0.272372
897,0.176821
1196,0.118097


## 3. Predictions for Submission


In [4]:
tok = AutoTokenizer.from_pretrained(FINAL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(DEVICE).eval()

preds, batch_size = [], 256
texts = test_df["text"].astype(str).tolist()
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], truncation=True, padding=True,
                  max_length=64, return_tensors="pt").to(DEVICE)
        preds.append(model(**enc).logits.argmax(-1).cpu().numpy())
predictions = np.concatenate(preds)

pred = test_df[["id"]].assign(label=predictions)
assert len(pred) == len(test_df) and (pred["id"].values == test_df["id"].values).all()
assert pred["label"].isin(LABELS).all()
pred.to_csv("pred_12.csv", index=False)

print(f"pred_12.csv written: {len(pred)} rows")
dist = pred["label"].value_counts(normalize=True).sort_index()
train_dist = train_df["label"].value_counts(normalize=True).sort_index()
print(pd.DataFrame({"train": train_dist, "predicted (test)": dist}).rename(index=LABEL_NAMES).round(3))


pred_12.csv written: 2388 rows
         train  predicted (test)
label                           
Bearish  0.151             0.157
Bullish  0.202             0.215
Neutral  0.647             0.628
